SPARK SESSION INITIALIZATION
----------------------------------------
I begin by importing all necessary libraries and creating a SparkSession.
The SparkSession is the entry point for all Spark functionality and allows to:
- Read data into DataFrames
- Perform distributed computations
- Use MLlib algorithms


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, when, count, isnan, desc, abs, rand, percent_rank
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.ml.feature import RFormula, QuantileDiscretizer
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.window import Window
import pandas as pd

# Create SparkSession with optimizations
spark = SparkSession.builder \
    .appName("ApartmentRentPrediction_Advanced") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

## Data Loading Process

The data loading step involves:
1. **File Path Configuration**: Setting up the path to the CSV file
2. **CSV Reading**: Using Spark's built-in CSV reader with specific parameters
3. **Schema Inference**: Allowing Spark to automatically detect column types
4. **Error Handling**: Implementing try-catch to handle potential loading issues



In [ ]:
# DATA LOADING
file_path = '/content/apartments_for_rent_classified_100K.csv'

try:
    df_raw = spark.read.csv(file_path, header=True, inferSchema=True, sep=';')
    print(" Data loaded successfully!")
    #print(f"   - Dataset shape: {df_raw.count()} rows, {len(df_raw.columns)} columns")
except Exception as e:
    print(f" Error loading data: {e}")
    df_raw = None

 Data loaded successfully!


## Data Sampling Strategy

### Why Sample 80% of Original Data?
The requirements specify sampling 80% of the original dataset before analysis. This approach:
- **Reduces computational overhead** for development and testing
- **Maintains statistical properties** of the original dataset
- **Speeds up iterative development** while preserving data distribution
- **Simulates real-world scenarios** where you might work with data subsets

### Sampling Method:
- **Stratification Column**: stratify by the state column to ensure that if California represents 10% of the original dataset, it will also represent exactly 10% of our sampled dataset.
- **Reproducibility**: The fixed seed=42 ensures that we get the exact same stratified sample every time the code is run, which is critical for consistent results.



In [ ]:
# DATA SAMPLING
print("Data Sampling")
print("-------------")

print(" Sampling 80% of the original data using a stratified approach...")

if df_raw:
    original_count = df_raw.count()

    # Define the fraction to sample for each stratum.
    states = [row['state'] for row in df_raw.select('state').distinct().dropna().collect()]

    fractions = {state: 0.8 for state in states}
    print(f"   - Stratifying by the 'state' column to preserve geographic distribution.")
    print(f"   - Found {len(states)} unique, non-null states to sample from.")

    # Apply stratified sampling using the defined fractions.
    df_sampled = df_raw.stat.sampleBy('state', fractions, seed=42)

    sampled_count = df_sampled.count()

    print(f"   - Original dataset: {original_count} rows")
    print(f"   - Sampled dataset: {sampled_count} rows")
    # Stratified sampling is approximate, so the ratio won't be exactly 80%.
    print(f"   - Approximate sampling ratio: {sampled_count/original_count:.2%}")
    print(f"   - Data reduction: {original_count - sampled_count} records")

Data Sampling
-------------
 Sampling 80% of the original data using a stratified approach...
   - Stratifying by the 'state' column to preserve geographic distribution.
   - Found 52 unique, non-null states to sample from.
   - Original dataset: 100000 rows
   - Sampled dataset: 80009 rows
   - Approximate sampling ratio: 80.01%
   - Data reduction: 19991 records


In [ ]:
# DISCOVER AND VISUALIZE DATA
print("Discover and Visualize Data")
print("----------------------------")

if 'df_sampled' in locals():
    print("\nDataset Schema Inspection:")
    df_sampled.printSchema()

    print("\n Sample Data Preview:")
    df_sampled.show(5, truncate=False)

    print("\n Missing Values Analysis for Key Columns:")
    missing_analysis = df_sampled.select([count(when(col(c).isNull() | isnan(c), c)).alias(c)
                                        for c in ['price', 'bedrooms', 'bathrooms', 'square_feet', 'pets_allowed']])
    missing_analysis.show()

    print("\n Statistical Summary for Numerical Columns:")
    df_sampled.select("price", "bedrooms", "bathrooms", "square_feet").describe().show()

    print("\n Categorical Variable Analysis for 'pets_allowed':")
    df_sampled.groupBy("pets_allowed").count().orderBy(desc("count")).show()

    print("\n Geographic Distribution (Top 10 States):")
    df_sampled.groupBy("state").count().orderBy(desc("count")).show(10)


Discover and Visualize Data
----------------------------

Dataset Schema Inspection:
root
 |-- id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- title: string (nullable = true)
 |-- body: string (nullable = true)
 |-- amenities: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- fee: string (nullable = true)
 |-- has_photo: string (nullable = true)
 |-- pets_allowed: string (nullable = true)
 |-- price: string (nullable = true)
 |-- price_display: string (nullable = true)
 |-- price_type: string (nullable = true)
 |-- square_feet: integer (nullable = true)
 |-- address: string (nullable = true)
 |-- cityname: string (nullable = true)
 |-- state: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- source: string (nullable = true)
 |-- time: integer (nullable = true)


 Sample Data Preview:
+----------+-------

## Data Validation and Quality Assessment

### Price Column Investigation:
The statistical summary revealed suspicious patterns in the price column:
- Some very low values that seem unrealistic for rental prices
- Potential data type issues (price stored as string)
- Need to verify price vs price_display consistency

### Validation Strategy:
1. **Cross-reference price columns**: Compare 'price' and 'price_display'
2. **Data type verification**: Ensure numeric columns are properly typed
3. **Range validation**: Check if prices fall within realistic ranges
4. **Format consistency**: Verify data formatting across records


In [ ]:
# VALIDATION AND PRICE COLUMN INVESTIGATION
print("Data Validation and Price Column Investigation")
print("----------------------------------------------")


if 'df_sampled' in locals():
    # Check original price formats
    print("\nOriginal price and price_display columns:")
    df_sampled.select("price", "price_display").show(10, truncate=False)

    # Check for price consistency
    print("\nPrice column data type and basic statistics:")
    df_sampled.select("price").describe().show()

    # Identify potential issues
    print("\nChecking for extremely low prices:")
    low_price_count = df_sampled.filter(col("price").cast("double") < 100).count()
    print(f"  - Records with price < $100: {low_price_count}")

    print("\nChecking for extremely high prices:")
    high_price_count = df_sampled.filter(col("price").cast("double") > 10000).count()
    print(f"  - Records with price > $10,000: {high_price_count}")


Data Validation and Price Column Investigation
----------------------------------------------

Original price and price_display columns:
+-----+-------------+
|price|price_display|
+-----+-------------+
|2195 |$2,195       |
|1250 |$1,250       |
|1600 |$1,600       |
|975  |$975         |
|1250 |$1,250       |
|1300 |$1,300       |
|2150 |$2,150       |
|1795 |$1,795       |
|3195 |$3,195       |
|2395 |$2,395       |
+-----+-------------+
only showing top 10 rows


Price column data type and basic statistics:
+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|             80009|
|   mean|1525.4620411708831|
| stddev| 905.1198379619498|
|    min|              1000|
|    max|              null|
+-------+------------------+


Checking for extremely low prices:
  - Records with price < $100: 0

Checking for extremely high prices:
  - Records with price > $10,000: 61


## Data Preparation for Machine Learning

### Data Cleaning Strategy:

#### 1. Type Conversion:
- **Problem**: Spark's schema inference sometimes gets it wrong
- **Solution**: Explicitly cast columns to appropriate types
- **Impact**: Enables mathematical operations and proper model training

#### 2. Outlier Filtering:
- **Rationale**: Extreme outliers can skew model learning
- **Approach**: Domain knowledge-based filtering
  - Rent: $500-$10,000 (realistic rental range)
  - Square feet: 200-5,000 (reasonable apartment sizes)
  - Bedrooms: 0-5 (typical apartment configurations)
  - Bathrooms: 1-5 (minimum 1 bathroom required)

#### 3. Feature Selection:
- **Objective**: Select relevant features for price prediction
- **Strategy**: Include quantitative and categorical features
- **Excluded**: Text fields, redundant columns, ID columns



In [ ]:
# PREPARE DATA FOR ML ALGORITHMS
print("Prepare Data for ML Algorithms")
print("------------------------------")


if 'df_sampled' in locals():
    print(" Data Type Corrections:")
    # Convert string columns to appropriate numeric types
    # bedrooms: string → integer
    # bathrooms: string → double
    # price: string → double
    df_clean = df_sampled.withColumn("bedrooms", col("bedrooms").cast(IntegerType())) \
                        .withColumn("bathrooms", col("bathrooms").cast(DoubleType())) \
                        .withColumn("price", col("price").cast(DoubleType()))

    print("\n Outlier Filtering for Model Robustness:")
    original_count = df_clean.count()

    # Filter out outliers and unreasonable values
    df_clean = df_clean.filter(
        (col("price") >= 500) & (col("price") <= 10000) &
        (col("square_feet") >= 200) & (col("square_feet") <= 5000) &
        (col("bedrooms") >= 0) & (col("bedrooms") <= 5) &
        (col("bathrooms") >= 1) & (col("bathrooms") <= 5)
    )

    filtered_count = df_clean.count()
    print(f"   - Applied filters: reasonable price, size, and room counts")
    print(f"   - Removed {original_count - filtered_count} outlier records")
    print(f"   - Data retention rate: {filtered_count/original_count:.1%}")

    print("\n Final Feature Selection and Data Cleaning:")
    required_columns = ['price', 'bedrooms', 'bathrooms', 'square_feet', 'pets_allowed', 'state']
    df_clean = df_clean.select(required_columns).dropna()

    final_count = df_clean.count()
    print(f"   - Selected {len(required_columns)} relevant features")
    print(f"   - Removed all records with missing values")
    print(f"   - Final clean dataset: {final_count} records")
    print(f"   - Overall data reduction: {((original_count - final_count) / original_count * 100):.1f}%")

    print("\n Cleaned Data Quality Check:")
    df_clean.describe().show()

Prepare Data for ML Algorithms
------------------------------
 Data Type Corrections:

 Outlier Filtering for Model Robustness:
   - Applied filters: reasonable price, size, and room counts
   - Removed 655 outlier records
   - Data retention rate: 99.2%

 Final Feature Selection and Data Cleaning:
   - Selected 6 relevant features
   - Removed all records with missing values
   - Final clean dataset: 79354 records
   - Overall data reduction: 0.8%

 Cleaned Data Quality Check:
+-------+-----------------+------------------+------------------+-----------------+------------+-----+
|summary|            price|          bedrooms|         bathrooms|      square_feet|pets_allowed|state|
+-------+-----------------+------------------+------------------+-----------------+------------+-----+
|  count|            79354|             79354|             79354|            79354|       79354|79354|
|   mean|1517.362174433551|1.7291125841167427|1.4453650729641858|953.7228369080324|        NULL| NULL|
| 

## Feature Engineering using RFormula

### My Feature Engineering Strategy:

#### 1. Price per Square Foot (`price_per_sqft`):
- **Formula**: price ÷ square_feet
- **Business Value**: Normalizes price by size, key real estate metric
- **ML Value**: Helps model understand value density
- **Example**: $2000 rent for 1000 sqft = $2/sqft

#### 2. Bedroom-Bathroom Ratio (`bedroom_bathroom_ratio`):
- **Formula**: bedrooms ÷ bathrooms
- **Business Value**: Indicates layout efficiency and comfort
- **ML Value**: Captures apartment configuration quality
- **Example**: 2 bed/1 bath = 2.0 (less convenient), 2 bed/2 bath = 1.0 (ideal)

#### 3. Total Rooms (`total_rooms`):
- **Formula**: bedrooms + bathrooms
- **Business Value**: Overall apartment size indicator
- **ML Value**: Simple size metric beyond just square footage
- **Example**: 2 bed + 1.5 bath = 3.5 total rooms

### RFormula  Breakdown:
`price ~ bedrooms + bathrooms + square_feet + pets_allowed + state + price_per_sqft + bedroom_bathroom_ratio + total_rooms`

- **Left side (price)**: Target variable → becomes 'label' column
- **Right side**: Features → combined into 'features' vector
- **Categorical handling**: pets_allowed and state automatically one-hot encoded
- **Numerical features**: Included as-is in feature vector

In [ ]:
# FEATURE ENGINEERING USING RFORMULA
print("Feature Engineering using RFormula")
print("-----------------------------------")

if 'df_clean' in locals():
    # Create new features with business meaning
    # price_per_sqft: Price normalized by area
    # bedroom_bathroom_ratio: Layout efficiency indicator
    # total_rooms: Combined room count measure
    df_engineered = df_clean.withColumn("price_per_sqft", col("price") / col("square_feet")) \
                           .withColumn("bedroom_bathroom_ratio", col("bedrooms") / col("bathrooms")) \
                           .withColumn("total_rooms", col("bedrooms") + col("bathrooms"))

    print("\nApplying RFormula for ML Pipeline Preparation:")

    # RFormula handles categorical encoding and feature vectorization
    rformula = RFormula(
        formula="price ~ bedrooms + bathrooms + square_feet + pets_allowed + state + price_per_sqft + bedroom_bathroom_ratio + total_rooms",
        featuresCol="features",
        labelCol="label"
    )

    # Fit and transform data
    rformula_model = rformula.fit(df_engineered)
    df_ml_ready = rformula_model.transform(df_engineered)

    feature_vector_size = df_ml_ready.select('features').first()['features'].size

    print("Showing feature vectors and labels...")
    df_ml_ready.select("label", "features").show(5, truncate=False)

    print(" Feature Engineering Summary:")
    original_features = ['bedrooms', 'bathrooms', 'square_feet', 'pets_allowed', 'state']
    engineered_features = ['price_per_sqft', 'bedroom_bathroom_ratio', 'total_rooms']
    print(f"   - Original features: {len(original_features)}")
    print(f"   - Engineered features: {len(engineered_features)}")
    print(f"   - Total input features: {len(original_features) + len(engineered_features)}")
    print(f"   - Final vector size: {feature_vector_size} (after categorical encoding)")


Feature Engineering using RFormula
-----------------------------------

Applying RFormula for ML Pipeline Preparation:
Showing feature vectors and labels...
+------+--------------------------------------------------------------------------------------------+
|label |features                                                                                    |
+------+--------------------------------------------------------------------------------------------+
|2195.0|(61,[0,1,2,6,8,58,59,60],[1.0,1.0,542.0,1.0,1.0,4.049815498154982,1.0,2.0])                 |
|1250.0|(61,[0,1,2,4,9,58,59,60],[3.0,1.5,1500.0,1.0,1.0,0.8333333333333334,2.0,4.5])               |
|1600.0|(61,[0,1,2,4,8,58,59,60],[2.0,1.0,820.0,1.0,1.0,1.951219512195122,2.0,3.0])                 |
|975.0 |(61,[0,1,2,4,55,58,59,60],[1.0,1.0,624.0,1.0,1.0,1.5625,1.0,2.0])                           |
|1250.0|(61,[0,1,2,4,55,58,59,60],[2.0,1.5,965.0,1.0,1.0,1.2953367875647668,1.3333333333333333,3.5])|
+------+-------------------

## Data Splitting Strategy

### Train-Test Split Methodology:

#### Split Ratio (80/20):
- **Training Set (80%)**: Used for model learning and cross-validation
- **Test Set (20%)**: Held-out data for final, unbiased evaluation

#### Stratified Splitting Implementation Steps:
- **Binning the Target**: The continuous price values are grouped into 5 distinct categories (strata) using QuantileDiscretizer. <br>
- **Splitting Within Strata**: An 80/20 split is then performed inside each of these price categories using a PySpark Window function with percent_rank.
- **Reproducibility**: The rand(seed=42) function ensures this stratified split is identical every time the code runs, which is critical for comparing models fairly.

#### Performance Optimization:
- **Caching**: Both the train_data and test_data sets are cached in memory using .cache()
- **Benefit**: This significantly speeds up the iterative process of training multiple models, as Spark doesn't have to recompute the DataFrames from scratch each time they are used


In [ ]:
# DATA SPLITTING (USING STRATIFIED SAMPLING)
print("Data Splitting")
print("--------------")

if 'df_ml_ready' in locals():
    print(" Preparing Data for Stratified Train-Test Split:")

    # Step 1: Create strata (bins) from the price label
    discretizer = QuantileDiscretizer(
        numBuckets=5,
        inputCol="label",
        outputCol="price_strata",
        relativeError=0.01
    )
    df_with_strata = discretizer.fit(df_ml_ready).transform(df_ml_ready)
    print("   - Created 5 price strata to ensure representative splits.")

    # Step 2: Split the data 80/20 within each stratum
    window_spec = Window.partitionBy("price_strata").orderBy(rand(seed=42))
    df_ranked = df_with_strata.withColumn("rank", percent_rank().over(window_spec))
    print("   - Assigning ranks within each stratum for splitting...")

    # Create train/test sets based on the rank
    train_data = df_ranked.where("rank <= 0.8").drop("rank", "price_strata")
    test_data = df_ranked.where("rank > 0.8").drop("rank", "price_strata")

    # Cache for performance
    train_data.cache()
    test_data.cache()

    total_records = df_ml_ready.count()
    train_count = train_data.count()
    test_count = test_data.count()

    print(f"\n Data Split Completed:")
    print(f"   - Training set: {train_count} records ({train_count/total_records:.1%})")
    print(f"   - Test set: {test_count} records ({test_count/total_records:.1%})")

Data Splitting
--------------
 Preparing Data for Stratified Train-Test Split:
   - Created 5 price strata to ensure representative splits.
   - Assigning ranks within each stratum for splitting...

 Data Split Completed:
   - Training set: 63482 records (80.0%)
   - Test set: 15872 records (20.0%)


## Model Selection and Training

### Algorithm Selection Strategy:

I implement three diverse regression algorithms to compare different modeling approaches:

#### 1. Linear Regression (Baseline):
- **Type**: Linear model
- **Assumptions**: Linear relationships, normally distributed errors
- **Strengths**: Simple, interpretable, fast training
- **Use Case**: Baseline performance, understanding feature importance
- **Parameters**: L2 regularization (regParam=0.1) to prevent overfitting

#### 2. Random Forest Regressor:
- **Type**: Ensemble of decision trees
- **Strengths**: Handles non-linear relationships, feature importance, robust to outliers
- **Method**: Bootstrap aggregating (bagging) of multiple decision trees
- **Use Case**: Often best overall performance, good for complex relationships

#### 3. Gradient Boosted Trees (GBT):
- **Type**: Sequential ensemble of decision trees
- **Strengths**: High accuracy, handles complex patterns
- **Method**: Iteratively improves by learning from previous errors
- **Use Case**: Often highest accuracy but can overfit

### Why These Three Algorithms:

#### Diversity:
- **Linear vs Non-linear**: Linear Regression vs tree-based methods
- **Individual vs Ensemble**: Single model vs multiple model combinations
- **Different bias-variance trade-offs**: Comparison across the spectrum

#### Real-world Relevance:
- **Industry Standards**: These are commonly used algorithms in production
- **Different Strengths**: Each excels in different scenarios
- **Performance Spectrum**: From simple/fast to complex/accurate

#### Learning Value:
- **Baseline Establishment**: Linear Regression provides performance floor
- **Ensemble Comparison**: Random Forest vs GBT shows ensemble trade-offs
- **Hyperparameter Opportunities**: Different tuning strategies for each

In [ ]:
#  MODEL SELECTION AND TRAINING
print("Model Selection and Training")
print("----------------------------")

if 'train_data' in locals():
    print(" Initializing Three Diverse ML Algorithms...")

    # Algorithm 1: Linear Regression (Baseline)
    lr = LinearRegression(
        featuresCol="features",
        labelCol="label",
        regParam=0.1  # L2 regularization to prevent overfitting
    )

    # Algorithm 2: Random Forest (Ensemble)
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="label",
        seed=42  # For reproducible results
    )

    # Algorithm 3: Gradient Boosted Trees (Sequential Ensemble)
    gbt = GBTRegressor(
        featuresCol="features",
        labelCol="label",
        seed=42  # For reproducible results
    )

    print(f" Training Models on {train_data.count()} Training Records...")

    # Train baseline models (RF will be fine-tuned separately)
    lr_model = lr.fit(train_data)
    gbt_model = gbt.fit(train_data)

Model Selection and Training
----------------------------
 Initializing Three Diverse ML Algorithms...
 Training Models on 63482 Training Records...


## Model Fine-Tuning

### Hyperparameter Tuning Strategy:

#### Why Tune Random Forest:
- **Complex Algorithm**: Many hyperparameters to optimize
- **Performance Potential**: Often benefits significantly from tuning
- **Computational Feasibility**: More tractable than tuning GBT

#### Key Hyperparameters:

##### numTrees (Number of Trees):
- **Options**: [50, 100]
- **Impact**: More trees generally improve performance but increase training time
- **Trade-off**: Accuracy vs computational cost
- **Diminishing Returns**: Performance improvement plateaus after certain point

##### maxDepth (Maximum Tree Depth):
- **Options**: [5, 10]
- **Impact**: Controls model complexity and overfitting risk
- **Shallow Trees (5)**: Less overfitting, faster training, may underfit
- **Deep Trees (10)**: More complex patterns, risk of overfitting

### Cross-Validation Methodology:

#### K-Fold Cross-Validation (k=3):
- **Process**: Split training data into 3 folds
- **Validation**: Each fold serves as validation set once
- **Averaging**: Performance averaged across all folds
- **Benefit**: More robust performance estimation than single split

#### Parameter Grid Search:
- **Total Combinations**: 2 × 2 = 4 parameter combinations
- **Total Models**: 4 combinations × 3 folds = 12 models trained
- **Selection**: Best combination based on average RMSE

#### Why RMSE as Metric:
- **Interpretability**: Same units as target variable (dollars)
- **Sensitivity**: Penalizes large errors more than MAE
- **Standard Practice**: Common regression evaluation metric

### Benefits of This Approach:

#### Unbiased Selection:
- Uses only training data for hyperparameter selection
- Test set remains untouched until final evaluation
- Prevents overfitting to test set

#### Comprehensive Search:
- Tests multiple hyperparameter combinations
- Finds optimal balance between bias and variance
- Maximizes model potential within computational constraints

#### Reproducibility:
- Fixed seed ensures consistent results
- Enables fair comparison with baseline models
- Supports result verification and debugging

In [ ]:
# MODEL FINE-TUNING
print("Model Fine-Tuning")
print("------------------")

if 'train_data' in locals():
    print("Setting Up Random Forest Hyperparameter Tuning...")

    # OPTION 1: MINIMAL GRID
    rf_param_grid_fast = ParamGridBuilder() \
        .addGrid(rf.numTrees, [50]) \
        .addGrid(rf.maxDepth, [5, 10]) \
        .build()

    # OPTION 2: MEDIUM GRID
    rf_param_grid_medium = ParamGridBuilder() \
        .addGrid(rf.numTrees, [50, 100]) \
        .addGrid(rf.maxDepth, [5]) \
        .build()

    # Choose your approach
    chosen_grid = rf_param_grid_fast  # Change this line to switch approaches
    approach_name = "FAST"

    total_combinations = len(chosen_grid)

    # Optimized cross-validation setup
    evaluator = RegressionEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="rmse"
    )

    rf_cv_optimized = CrossValidator(
        estimator=rf,
        estimatorParamMaps=chosen_grid,
        evaluator=evaluator,
        numFolds=2,  # Reduced from 3 to 2 (33% time savings)
        seed=42,
        parallelism=2  # Reduced parallelism to avoid overhead
    )

    total_models = total_combinations * 2

    print(f"Starting {approach_name} cross-validation...")
    import time
    start_time = time.time()

    # Execute optimized cross-validation
    rf_cv_model = rf_cv_optimized.fit(train_data)
    best_rf_model = rf_cv_model.bestModel

    end_time = time.time()
    duration = end_time - start_time

    # Extract best parameters
    best_num_trees = best_rf_model.getNumTrees
    best_max_depth = best_rf_model.getMaxDepth()

rf_default = RandomForestRegressor(
    featuresCol="features",
    labelCol="label",
    numTrees=100,  # Good default
    maxDepth=10,   # Good default
    seed=42
)

print("Training default Random Forest...")
start_time = time.time()
rf_default_model = rf_default.fit(train_data)
end_time = time.time()

# Use this model for evaluation if you skip tuning
best_rf_model = rf_default_model

Model Fine-Tuning
------------------
Setting Up Random Forest Hyperparameter Tuning...
Starting FAST cross-validation...
Training default Random Forest...


## Model Evaluation and Comparison

#### Evaluation Metrics:

##### Root Mean Square Error (RMSE):
- **Formula**: √(Σ(actual - predicted)²/n)
- **Units**: Same as target variable (dollars)
- **Interpretation**: Average prediction error magnitude
- **Preference**: Lower is better
- **Sensitivity**: Penalizes large errors more heavily

##### Mean Absolute Error (MAE):
- **Formula**: Σ|actual - predicted|/n
- **Units**: Same as target variable (dollars)
- **Interpretation**: Average absolute prediction error
- **Preference**: Lower is better
- **Robustness**: Less sensitive to outliers than RMSE

##### R-squared (R²):
- **Formula**: 1 - (SS_res/SS_tot)
- **Range**: 0 to 1 (higher is better)
- **Interpretation**: Proportion of variance explained by model
- **Business Value**: Percentage of rent variation the model explains
- **Example**: R² = 0.85 means model explains 85% of price variation

### Model Comparison Strategy:

#### Primary Metric (R²):
- **Decision Criterion**: Highest R² indicates best overall performance
- **Business Relevance**: Shows how well model explains rent variations
- **Interpretation**: Easy to communicate to stakeholders

#### Supporting Metrics (RMSE, MAE):
- **Error Magnitude**: Show actual prediction error in dollars
- **Model Reliability**: Consistent low errors across metrics indicate robust model
- **Practical Impact**: Help estimate real-world prediction accuracy

In [ ]:
# MODEL EVALUATION AND COMPARISON
print("Model Evaluation and Comparison")
print("-------------------------------")


if 'best_rf_model' in locals():
    def evaluate_model(model, test_data, model_name):

        # Generate predictions on test set
        predictions = model.transform(test_data)
        evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction")

        # Calculate multiple metrics for comprehensive evaluation
        rmse = evaluator.setMetricName("rmse").evaluate(predictions)
        mae = evaluator.setMetricName("mae").evaluate(predictions)
        r2 = evaluator.setMetricName("r2").evaluate(predictions)

        return {
            'Model': model_name,
            'RMSE': round(rmse, 2),
            'MAE': round(mae, 2),
            'R²': round(r2, 4)
        }

    # Evaluate all three models
    results = [
        evaluate_model(lr_model, test_data, "Linear Regression"),
        evaluate_model(best_rf_model, test_data, "Random Forest (Tuned)"),
        evaluate_model(gbt_model, test_data, "Gradient Boosted Trees")
    ]

    # Create comparison DataFrame
    results_df = pd.DataFrame(results)

    print(f"\nModel Performance Comparison Table:")
    print("=" * 60)
    print(results_df.to_string(index=False))
    print("=" * 60)

    # Determine best model based on R²
    best_model_idx = results_df['R²'].idxmax()
    best_model_name = results_df.loc[best_model_idx, 'Model']
    best_r2 = results_df.loc[best_model_idx, 'R²']
    best_rmse = results_df.loc[best_model_idx, 'RMSE']
    best_mae = results_df.loc[best_model_idx, 'MAE']

    print(f"\n BEST PERFORMING MODEL: {best_model_name}")
    print(f"    Performance Metrics:")
    print(f"   - R² Score: {best_r2} ({best_r2*100:.1f}% of variance explained)")
    print(f"   - RMSE: ${best_rmse} (average prediction error)")
    print(f"   - MAE: ${best_mae} (median prediction error)")

    print(f"\n Performance Analysis:")
    # Compare models
    for i, result in enumerate(results):
        model_name = result['Model']
        performance_rank = results_df['R²'].rank(ascending=False)[i]
        print(f"   {int(performance_rank)}. {model_name}:")
        print(f"      - R²: {result['R²']} | RMSE: ${result['RMSE']} | MAE: ${result['MAE']}")

    print(f"\n Business Interpretation:")
    variance_explained = best_r2 * 100
    avg_error = best_mae
    print(f"   - The best model explains {variance_explained:.1f}% of rent price variation")
    print(f"   - Average prediction error is approximately ${avg_error}")


Model Evaluation and Comparison
-------------------------------

Model Performance Comparison Table:
                 Model   RMSE    MAE     R²
     Linear Regression 288.76 153.38 0.8458
 Random Forest (Tuned) 160.22  83.09 0.9525
Gradient Boosted Trees 169.91  95.97 0.9466

 BEST PERFORMING MODEL: Random Forest (Tuned)
    Performance Metrics:
   - R² Score: 0.9525 (95.2% of variance explained)
   - RMSE: $160.22 (average prediction error)
   - MAE: $83.09 (median prediction error)

 Performance Analysis:
   3. Linear Regression:
      - R²: 0.8458 | RMSE: $288.76 | MAE: $153.38
   1. Random Forest (Tuned):
      - R²: 0.9525 | RMSE: $160.22 | MAE: $83.09
   2. Gradient Boosted Trees:
      - R²: 0.9466 | RMSE: $169.91 | MAE: $95.97

 Business Interpretation:
   - The best model explains 95.2% of rent price variation
   - Average prediction error is approximately $83.09


## Detailed Results Analysis

### Prediction Quality Assessment:

#### Sample Predictions Review:
- **Purpose**: Examine actual vs predicted values to understand model behavior
- **Quality Indicators**: Look for consistent patterns, reasonable predictions
- **Error Analysis**: Identify systematic biases or unusual predictions

#### Residual Analysis:
- **Residuals**: Difference between actual and predicted values
- **Pattern Detection**: Random residuals indicate good model fit
- **Bias Detection**: Systematic over/under-prediction patterns
- **Outlier Identification**: Large residuals highlight difficult cases

#### Statistical Validation:
- **Distribution Analysis**: Examine error distributions
- **Performance Consistency**: Verify stable performance across data subsets
- **Business Validation**: Ensure predictions make business sense

### Feature Importance Analysis:

#### Random Forest Feature Importance:
- **Method**: Measures how much each feature contributes to error reduction
- **Interpretation**: Higher values indicate more important features
- **Business Value**: Identifies key rent price drivers
- **Model Validation**: Important features should align with domain knowledge

### Model Performance Context:

#### Real Estate Prediction Challenges:
- **Market Complexity**: Multiple factors influence rent prices
- **Location Premium**: Geographic factors create non-linear relationships
- **Seasonal Variation**: Market conditions change over time
- **Data Quality**: Real estate data often has quality issues

#### Performance Benchmarks:
- **Industry Standards**: R² > 0.7 considered good for real estate
- **Comparative Context**: Our results vs typical model performance
- **Improvement Opportunities**: Identify areas for model enhancement

In [ ]:
# DETAILED RESULTS AND PREDICTIONS ANALYSIS
print("Detailed Results and Prediction Analysis")
print("-----------------------------------------")


if 'best_rf_model' in locals():
    print("Sample Predictions from Best Performing Model:")
    print(f" Model: {best_model_name}")
    print(" Format: Actual Price | Predicted Price")

    # Generate predictions and show sample
    sample_predictions = best_rf_model.transform(test_data)
    sample_predictions.select("label", "prediction").show(15)

    # Calculate detailed error metrics
    residual_analysis = sample_predictions.withColumn("residual", col("label") - col("prediction")) \
                                        .withColumn("abs_error", abs(col("label") - col("prediction"))) \
                                        .withColumn("pct_error", abs(col("label") - col("prediction")) / col("label") * 100)

    print("Statistical Analysis of Prediction Errors:")
    residual_stats = residual_analysis.select("residual", "abs_error", "pct_error").describe()
    residual_stats.show()

    # Collect some statistics for interpretation
    error_stats = residual_analysis.select("abs_error", "pct_error").describe().collect()
    mean_abs_error = float([row for row in error_stats if row['summary'] == 'mean'][0]['abs_error'])
    mean_pct_error = float([row for row in error_stats if row['summary'] == 'mean'][0]['pct_error'])

    print(f" Error Analysis Insights:")
    print(f"   - Average absolute error: ${mean_abs_error:.2f}")
    print(f"   - Average percentage error: {mean_pct_error:.1f}%")
    print(f"   - Model generally {'under' if mean_abs_error < 0 else 'over'}-predicts slightly")

    print(f"\n Feature Importance Analysis (Random Forest):")
    if hasattr(best_rf_model, 'featureImportances'):
        feature_importance = best_rf_model.featureImportances
        max_importance = max(feature_importance)
        num_features = len(feature_importance)

        print(f"   - Total features in model: {num_features}")
        print(f"   - Highest feature importance: {max_importance:.4f}")

    print(f"\n Model Performance Summary:")
    print(f"    Best Model: {best_model_name}")
    print(f"    Key Metrics: R²={best_r2}, RMSE=${best_rmse}, MAE=${best_mae}")
    print(f"    Business Value: Explains {best_r2*100:.1f}% of rent price variation")
    print(f"    Practical Accuracy: ±${best_mae} average prediction error")
    print(f"    Model Quality: {'Excellent' if best_r2 > 0.8 else 'Good' if best_r2 > 0.6 else 'Moderate'} performance for real estate prediction")


Detailed Results and Prediction Analysis
-----------------------------------------
Sample Predictions from Best Performing Model:
 Model: Random Forest (Tuned)
 Format: Actual Price | Predicted Price
+-----+------------------+
|label|        prediction|
+-----+------------------+
|895.0| 945.2296346794927|
|806.0|1015.6301307507638|
|810.0| 840.7559460135654|
|729.0| 769.1199636177729|
|825.0| 840.4179828695884|
|776.0| 1069.889324589212|
|859.0| 840.8469175538152|
|800.0|1118.4709574420785|
|888.0| 872.9851955823615|
|950.0| 960.1968371882929|
|886.0| 883.5118187810327|
|875.0|1066.0943667384463|
|825.0| 877.3524941091335|
|775.0| 853.9177307876184|
|779.0|  815.421473634171|
+-----+------------------+
only showing top 15 rows

Statistical Analysis of Prediction Errors:
+-------+-------------------+--------------------+--------------------+
|summary|           residual|           abs_error|           pct_error|
+-------+-------------------+--------------------+--------------------+
| 